In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("F1_OneHotEncoder_Demo") \
    .getOrCreate()

print("Spark session successfully created.")

26/06/11 10:22:02 WARN Utils: Your hostname, Abhisheks-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.22.74.127 instead (on interface en0)
26/06/11 10:22:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 10:22:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session successfully created.


In [3]:
# Load the CSV file into a DataFrame
df = spark.read.csv("f1_engines.csv", header=True, inferSchema=True)

print("Original F1 Data:")
df.show()

Original F1 Data:
+---------------+---------------+
|    driver_name|engine_supplier|
+---------------+---------------+
| Max Verstappen|     Honda RBPT|
|Charles Leclerc|        Ferrari|
| George Russell|       Mercedes|
|   Lando Norris|       Mercedes|
|Fernando Alonso|       Mercedes|
|   Pierre Gasly|        Renault|
|   Yuki Tsunoda|     Honda RBPT|
|Valtteri Bottas|        Ferrari|
|Nico Hulkenberg|        Ferrari|
|Alexander Albon|       Mercedes|
+---------------+---------------+



In [4]:
# Step 1: Convert the categorical string column into numerical indices
indexer = StringIndexer(
    inputCol="engine_supplier", 
    outputCol="engine_index"
)

In [5]:
# Fit and transform the data
indexed_df = indexer.fit(df).transform(df)

print("After String Indexing:")
indexed_df.show()

After String Indexing:
+---------------+---------------+------------+
|    driver_name|engine_supplier|engine_index|
+---------------+---------------+------------+
| Max Verstappen|     Honda RBPT|         2.0|
|Charles Leclerc|        Ferrari|         1.0|
| George Russell|       Mercedes|         0.0|
|   Lando Norris|       Mercedes|         0.0|
|Fernando Alonso|       Mercedes|         0.0|
|   Pierre Gasly|        Renault|         3.0|
|   Yuki Tsunoda|     Honda RBPT|         2.0|
|Valtteri Bottas|        Ferrari|         1.0|
|Nico Hulkenberg|        Ferrari|         1.0|
|Alexander Albon|       Mercedes|         0.0|
+---------------+---------------+------------+



In [6]:
# Step 2: Convert the numerical indices into One-Hot Encoded vectors
encoder = OneHotEncoder(
    inputCols=["engine_index"],
    outputCols=["engine_vector"]
)

In [7]:
# Fit and transform the data
encoded_df = encoder.fit(indexed_df).transform(indexed_df)

print("After One-Hot Encoding:")
encoded_df.show(truncate=False)

After One-Hot Encoding:
+---------------+---------------+------------+-------------+
|driver_name    |engine_supplier|engine_index|engine_vector|
+---------------+---------------+------------+-------------+
|Max Verstappen |Honda RBPT     |2.0         |(3,[2],[1.0])|
|Charles Leclerc|Ferrari        |1.0         |(3,[1],[1.0])|
|George Russell |Mercedes       |0.0         |(3,[0],[1.0])|
|Lando Norris   |Mercedes       |0.0         |(3,[0],[1.0])|
|Fernando Alonso|Mercedes       |0.0         |(3,[0],[1.0])|
|Pierre Gasly   |Renault        |3.0         |(3,[],[])    |
|Yuki Tsunoda   |Honda RBPT     |2.0         |(3,[2],[1.0])|
|Valtteri Bottas|Ferrari        |1.0         |(3,[1],[1.0])|
|Nico Hulkenberg|Ferrari        |1.0         |(3,[1],[1.0])|
|Alexander Albon|Mercedes       |0.0         |(3,[0],[1.0])|
+---------------+---------------+------------+-------------+



In [8]:
# Clean up and stop the Spark session
spark.stop()
print("Spark session stopped.")

Spark session stopped.
